# Step 6: RAG with Microsoft Agent Framework + Foundry

Same RAG agent as Step 5, but uses **FoundryChatClient** from `agent-framework-foundry` to connect directly to an Azure AI Foundry project endpoint instead of a standalone Azure OpenAI resource.

References:
- [Agent Framework Overview](https://learn.microsoft.com/en-us/agent-framework/overview/?pivots=programming-language-python)
- [Foundry Provider](https://learn.microsoft.com/en-us/agent-framework/agents/providers/microsoft-foundry?pivots=programming-language-python)
- [Azure-Samples/python-agentframework-demos](https://github.com/Azure-Samples/python-agentframework-demos)

In [1]:
%pip install agent-framework agent-framework-foundry azure-identity azure-search-documents python-dotenv openai -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential as SyncDefaultAzureCredential, get_bearer_token_provider as sync_get_bearer_token_provider
from azure.identity.aio import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery, QueryType
from openai import OpenAI
from pydantic import Field
from dotenv import load_dotenv

load_dotenv(override=False)

# ── Foundry project config ──
FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL = os.getenv("FOUNDRY_MODEL")

# ── Async credential (for FoundryChatClient) ──
credential = DefaultAzureCredential()

# ── Foundry chat client ──
chat_client = FoundryChatClient(
    project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=credential,
)

# ── Sync credential (for embeddings + search — both sync clients) ──
sync_credential = SyncDefaultAzureCredential()
sync_token_provider = sync_get_bearer_token_provider(
    sync_credential, "https://cognitiveservices.azure.com/.default"
)

# ── Embedding client (sync OpenAI SDK) ──
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
EMBEDDING_DIMS = 256  # Must match what was used in Step 3
embedding_client = OpenAI(
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=sync_token_provider,
)

# ── Azure AI Search client (sync) ──
search_client = SearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-index"),
    credential=sync_credential,
)

print(f"Foundry endpoint: {FOUNDRY_PROJECT_ENDPOINT}")
print(f"Foundry model: {FOUNDRY_MODEL}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT} (dims={EMBEDDING_DIMS})")
print(f"Search index: {os.getenv('AZURE_SEARCH_INDEX_NAME')}")
print("Ready!")

Foundry endpoint: https://admin-megt79wf-eastus2.services.ai.azure.com/api/projects/admin-megt79wf-eastus2-project
Foundry model: gpt-4.1
Embedding model: text-embedding-3-large-460208 (dims=256)
Search index: rag-index
Ready!


## Define the search tool

Same hybrid search + semantic reranking tool as Step 5. The `@tool` decorator turns it into a function tool the agent calls autonomously.

In [3]:
@tool(name="search_books", description="Search the Harry Potter books for information. Returns relevant passages with page numbers.")
def search_books(
    query: Annotated[str, Field(description="The search query to find relevant passages in the Harry Potter books.")],
    top_k: Annotated[int, Field(description="Number of results to return.")] = 5,
) -> str:
    """Hybrid search + semantic reranking over the Harry Potter book index."""
    # Embed the query
    response = embedding_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=[query],
        dimensions=EMBEDDING_DIMS,
    )
    query_vector = response.data[0].embedding

    # Hybrid search with semantic reranking
    results = search_client.search(
        search_text=query,
        vector_queries=[
            VectorizedQuery(vector=query_vector, k=top_k, fields="embedding")
        ],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="default-semantic",
        top=top_k,
    )

    # Format results for the agent
    chunks = []
    for r in results:
        score_info = f"(reranker: {r.get('@search.reranker_score', 0):.2f})"
        chunks.append(f"[Page {r['page_number']}] {score_info}\n{r['content']}")

    if not chunks:
        return "No relevant passages found."

    return "\n\n---\n\n".join(chunks)

print("Search tool defined.")

Search tool defined.


## Create the Foundry Agent

Uses `FoundryChatClient` connected to your Azure AI Foundry project. The agent owns instructions, tools, and session handling locally.

In [4]:
agent = Agent(
    client=chat_client,
    name="HarryPotterRAG-Foundry",
    instructions=(
        "You are a helpful assistant that answers questions about the Harry Potter books. "
        "Use the search_books tool to find relevant passages before answering. "
        "Always cite page numbers in your answer like (Page X). "
        "If the search results don't contain enough information, say so."
    ),
    tools=[search_books],
)

print(f"Agent '{agent.name}' created with tool: search_books")

Agent 'HarryPotterRAG-Foundry' created with tool: search_books


## Ask questions

In [5]:
result = await agent.run(
    "What are the different names that the Dark Lord had in the book? Tell me in which parts of the book these names come up as well."
)
print(result.text)

The Dark Lord, also known as Lord Voldemort, is referred to by several different names in the Harry Potter books. Below are the main names used and examples of where they occur:

1. **The Dark Lord**  
- This is the title most commonly used by Voldemort’s followers (Death Eaters) and those who fear his real name.
  - Example: “And you thought, just for a laugh, you’d use the Dark Lord’s name?” (Page 3358)
  - “He does not need protection. The Dark Lord has gone— The Dark Lord will return...” (Page 3555)
  - Multiple references in conversations with Snape, Narcissa, and Bellatrix (Pages 2441, 2436, 2439, 2442, 2443)

2. **Lord Voldemort**
- This is his chosen name, used both by his followers and others (with varying degrees of fear or respect).
  - Example: “Rather like you, Severus. Weren’t you hoping that Lord Voldemort would spare her?” (Page 3555)
  - “I refer to the plan Lord Voldemort is revolving around me.” (Page 3558)
  - “Voldemort was there... Lord Voldemort...” (Page 1509)



In [6]:
result = await agent.run(
    "In Chamber of Secrets, how is Voldemort's name and how does he name himself?"
)
print(result.text)

In *Chamber of Secrets*, Voldemort's real name is revealed to be Tom Marvolo Riddle. He explains to Harry that he rearranged the letters of his name to create a new identity: "I AM LORD VOLDEMORT." Riddle says, “It was a name I was already using at Hogwarts, to my most intimate friends only, of course. You think I was going to use my filthy Muggle father’s name forever?... No, Harry — I fashioned myself a new name, a name I knew wizards everywhere would one day fear to speak," showing that he created the name "Voldemort" to distance himself from his Muggle heritage and inspire fear (Page 542).


In [7]:
result = await agent.run(
    "Who is Harry's godfather and how is he related to his parents?"
)
print(result.text)

Harry's godfather is Sirius Black. Sirius was his mum and dad’s best friend (Page 939). Additionally, Sirius bought Harry his first broomstick, showing he was close to Harry's parents, James and Lily Potter (Page 3127). This close friendship was the basis for his role as Harry's godfather.


## Multi-turn conversation

Session keeps conversation history so the agent handles follow-up questions.

In [9]:
session = agent.create_session()

# Turn 1
response = await agent.run("What is the Philosopher's Stone can you describe it?", session=session)
print(f"Turn 1:\n{response.text}\n")

# Turn 2 — follow-up
response = await agent.run("Who was trying to steal it and why?", session=session)
print(f"Turn 2:\n{response.text}\n")

Turn 1:
The search did not return a clear textual description of the Philosopher's Stone. Based on the books, the Philosopher's Stone is a legendary magical object that can turn any metal into pure gold and produces the Elixir of Life, granting immortality. It is typically described as a small, red stone that looks like a ruby (Page 219). However, if you need more detail or a direct passage from the book, the current search results do not provide that explicit description. If you want more information or a passage citing its appearance, let me know!

Turn 2:
Professor Quirrell was the one who tried to steal the Philosopher's Stone. He was acting under the orders and influence of Lord Voldemort, who possessed Quirrell's body to closely supervise the attempt. Voldemort wanted the stone to regain his physical form and to achieve immortality through the Elixir of Life. The plan failed because Harry Potter thwarted them and prevented Voldemort from obtaining the Stone (Page 1494, Page 263).

## Streaming

In [10]:
print("Agent: ", end="", flush=True)
stream = agent.run("Describe the Triwizard Tournament.", stream=True)
async for chunk in stream:
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()
await stream.get_final_response()

Agent: The Triwizard Tournament is a magical competition established about seven hundred years ago between the three largest European schools of wizardry: Hogwarts, Beauxbatons, and Durmstrang. A champion is selected from each school, and these three champions compete in three magical tasks that test their magical skill, daring, intelligence, and ability to face danger. The tournament is hosted by each school in turns, and was originally intended as a friendly way to promote magical cooperation and camaraderie among the schools (Page 1102).

The tasks are spaced throughout the school year, and champions are scored on their performance in each task. The champion with the highest total score after all three tasks wins the Triwizard Cup (Page 1160).

At the start of the tournament described in Harry Potter and the Goblet of Fire, the delegations from Beauxbatons and Durmstrang arrive at Hogwarts, the host school for that year (Page 1142). The tasks themselves are kept secret until shortly

## Cleanup

In [ ]:
await credential.close()
print("Done!")